# Slotfill - phase 3: LoRA training on Colab or Kaggle

Thin wrapper around `python -m train.train_vlm`. All the logic lives in the repo; this notebook
only installs, fetches the data bundle, runs the script and keeps the results somewhere that
survives the session. The same notebook runs on both platforms: set `PLATFORM` in the first cell.

**Runtime.** A T4 (free on both) is enough for Qwen2.5-VL-3B in 4-bit: expect roughly 1-2 hours
for one epoch over 1,750 pages, ~2.3x that for the `train_4k` ablation. Training checkpoints
every 50 steps and **a rerun of the same cell resumes** from the last checkpoint, so a lost
session costs minutes, not the run.

**Before running:** `python -m train.export` locally, then upload `slotfill-bundle.tar.gz` (~135 MB).

| | Colab | Kaggle |
|---|---|---|
| Bundle goes to | Google Drive, `MyDrive/slotfill/slotfill-bundle.tar.gz` | a private Kaggle dataset (Add Input -> Upload), slug `slotfill-bundle` |
| Output lives in | Drive, `MyDrive/slotfill/adapters/` (persists on disconnect) | `/kaggle/working/adapters/` (persists only via *Save Version -> Save & Run All*, which also runs unattended for up to 12h) |
| Needs | GPU runtime (Runtime -> Change runtime type -> T4) | phone-verified account, *Internet on* and *GPU T4* in the notebook settings |
| Watch out for | idle disconnects: keep the tab open | interactive sessions lose `/kaggle/working` on a kernel crash; use Save & Run All for the long ablation |

In [ ]:
PLATFORM = 'colab'   # or 'kaggle'

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content'
    BUNDLE_ARCHIVE = '/content/drive/MyDrive/slotfill/slotfill-bundle.tar.gz'
    OUT_ROOT = '/content/drive/MyDrive/slotfill/adapters'   # checkpoints survive a disconnect here
elif PLATFORM == 'kaggle':
    ROOT = '/kaggle/working'
    BUNDLE_ARCHIVE = '/kaggle/input/slotfill-bundle/slotfill-bundle.tar.gz'
    OUT_ROOT = '/kaggle/working/adapters'
else:
    raise ValueError(PLATFORM)
print(ROOT, BUNDLE_ARCHIVE, OUT_ROOT)

In [ ]:
%%capture
# Unsloth's recommended install. If it fights the preinstalled torch, fall back to the install
# cell of the current official Unsloth vision notebook for this platform (they differ).
!pip install unsloth

In [ ]:
import os
import subprocess

REPO = 'https://github.com/InaPD/invoiception.git'
BRANCH = 'lora-training-phase3'   # or master once merged
repo_dir = f'{ROOT}/invoiception'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, repo_dir], check=True)
%cd {repo_dir}
!git pull --ff-only
!pip install -q -e . --no-deps && pip install -q jsonschema pillow
!git log --oneline -1

In [ ]:
# Unpack the bundle onto the fast local disk (the images are read many times).
!mkdir -p {ROOT}/data && tar -xzf "{BUNDLE_ARCHIVE}" -C {ROOT}/data
!ls {ROOT}/data/slotfill-bundle && head -c 400 {ROOT}/data/slotfill-bundle/bundle.json
BUNDLE = f'{ROOT}/data/slotfill-bundle'

## Shakedown: 16 examples, a few steps

Proves the pipeline end to end (model loads, collator works, checkpoint writes, adapter saves,
smoke test runs) before spending an hour on the real run. Ignore its smoke numbers.

In [ ]:
!python -m train.train_vlm --bundle {BUNDLE} --split train \
    --out {OUT_ROOT}/shakedown --limit 16 --save-steps 1 --smoke-n 3 --fresh

## The main run: r=16, alpha=16, lr=2e-4, 1 epoch, 1,750 documents

If the session dies, rerun this exact cell: it resumes from `trainer/checkpoint-N` in `--out`.

In [ ]:
!python -m train.train_vlm --bundle {BUNDLE} --split train \
    --out {OUT_ROOT}/qwen25vl3b-r16-train --r 16 --alpha 16 --lr 2e-4 --epochs 1

## Ablation: data mix, 1,750 vs 4,025 documents (same 35 layouts, same knobs)

~2.3x the main run. On Kaggle, prefer *Save Version -> Save & Run All* so it finishes unattended.

In [ ]:
!python -m train.train_vlm --bundle {BUNDLE} --split train_4k \
    --out {OUT_ROOT}/qwen25vl3b-r16-train4k --r 16 --alpha 16 --lr 2e-4 --epochs 1

## What to bring back to the repo

For each finished adapter under `OUT_ROOT/<name>/`:

- `run_config.json`, `smoke.json`, `train_log.json` -> commit under `runs/train/<name>/`
- `adapter/` -> keep out of git (`adapters/<name>/adapter/` locally, or push to a private HF repo
  so vLLM can pull it by name in phase 5)
- `trainer/` (checkpoints) -> delete once the adapter is saved; it is only there for resuming.

In [ ]:
!ls -la {OUT_ROOT}/*/ && du -sh {OUT_ROOT}/*/adapter

In [ ]:
# Optional: push an adapter to the Hugging Face Hub (private repo). Colab: add HF_TOKEN as a
# secret and enable notebook access. Kaggle: Add-ons -> Secrets.
# from huggingface_hub import HfApi, login
# login(token=os.environ['HF_TOKEN'])
# HfApi().upload_folder(folder_path=f'{OUT_ROOT}/qwen25vl3b-r16-train/adapter',
#                       repo_id='<user>/slotfill-qwen25vl3b-r16', repo_type='model', private=True)